This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path
from enum import Enum

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences, savgol_filter

from data_processing.dot_env import config
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide

### Functions

In [ ]:
# phi_flat = phi.counts.reshape(-1)
# phi_mids = phi.midpoints[1]
def get_graphable_data(phi: NDHistogram) -> tuple[np.ndarray, np.ndarray]:
    """Get data from phi in graphable format.

    Counts and energy bin midpoints will be extracted as 1D arrays.
    :param phi: Neutron energy spectrum
    :type phi: NDHistogram
    :return: Counts array, energy bin midpoints array
    :rtype: tuple[np.ndarray, np.ndarray]
    """
    return phi.counts.reshape(-1), phi.midpoints[-1]

## Setting Entry

In [ ]:
energies = np.arange(0.124, 6.014, step=0.062)
exp_data = {f"{energy:.3f}_MeV": {} for energy in energies}

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
# bins_max value determined by detector energy calibration range
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
    1.2,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.01 MeVee)",
    0.01,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

## Data Processing

### Loading

In [ ]:
main_data_folder = Path(config["NEUTRON_DATA_FOLDER"])

R = load_neutron_response_matrix(
    main_data_folder / "response_matrix_R4_mono",
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
bins = np.arange(bins_min, bins_max + bins_width, bins_width)

base_path = main_data_folder / "response_matrix_R4_mono"
for energy, eng_data in exp_data.items():
    filepath = base_path / f"neutron_{energy}.csv.npy"
    eng_data["filepath"] = filepath
    L_array = np.load(filepath)

    np_cps, *_ = np.histogram(L_array, bins=bins)

    mov_avg_window = 7
    polyorder = 3
    np_cps_savgol = savgol_filter(
        np_cps, window_length=mov_avg_window, polyorder=polyorder
    )

    np_cps = np_cps.reshape(-1, 1)
    np_cps_savgol = np_cps_savgol.reshape(-1, 1)
    np_Ls = (bins[1:] + bins[:-1]) / 2

    if energy == "1.302_MeV":
        pass

    # N = NDHistogram(np_cps, [np_Ls, np.ones(1)])
    N = NDHistogram(np_cps_savgol, [np_Ls, np.ones(1)])
    eng_data["N"] = N

### Processing

In [ ]:
for energy, eng_data in exp_data.items():
    print(energy)
    N = eng_data["N"]
    phi, unfold_info = unfold_spectrum(
        R,
        N,
        L_cut=0.05,
        # tolerance=0.0000001,
        max_iterations=1000,
        full_info=True
    )
    eng_data["phi"] = phi
    eng_data["unfold_info"] = unfold_info

In [ ]:
for energy, eng_data in exp_data.items():
    errors = eng_data["unfold_info"]["errors"]
    iters = len(errors)
    print(f"{energy}: {iters} iters.")

### Uncertainty Estimation

#### Horizontal

In [ ]:
sigma = 0.050   # MeVee

In [ ]:
lshift_R = load_neutron_response_matrix(
    # Path("response_matrix_hi_res"),
    # Path("response_matrix_4k"),
    Path("response_matrix_R4_mono"),
    # min_L=-3*sigma,
    min_L=0,
    max_L=bins_max-3*sigma,
    L_bin_widths=bins_width
)
for energy, eng_data in exp_data.items():
    N = eng_data["N"]
    lshift_mids = N.midpoints[0] - 3*sigma
    nonzero_mask = lshift_mids > 0
    lshift_N = NDHistogram(N.counts[nonzero_mask], [lshift_mids[nonzero_mask], N.midpoints[1]])
    # lshift_N = NDHistogram(N.counts, [lshift_mids, N.midpoints[1]])

    if np.isclose(lshift_N.midpoints[0], lshift_R.midpoints[0]).all():
        lshift_N = NDHistogram(lshift_N.counts, [lshift_R.midpoints[0], lshift_N.midpoints[1]])
    else:
        raise ValueError("R and N midpoints don't match")

    lshift_phi, *_ = unfold_spectrum(
        lshift_R,
        lshift_N,
        max_iterations=2000
    )
    eng_data["lshift_N"] = lshift_N
    eng_data["lshift_phi"] = lshift_phi

In [ ]:
rshift_R = load_neutron_response_matrix(
    # Path("response_matrix_hi_res"),
    # Path("response_matrix_4k"),
    Path("response_matrix_R4_mono"),
    min_L=bins_min+3*sigma,
    max_L=bins_max+3*sigma,
    L_bin_widths=bins_width
)

for energy, eng_data in exp_data.items():
    N = eng_data["N"]
    rshift_mids = N.midpoints[0] + 3*sigma
    rshift_N = NDHistogram(N.counts, [rshift_mids, N.midpoints[1]])

    if np.isclose(rshift_N.midpoints[0], rshift_R.midpoints[0]).all():
        rshift_N = NDHistogram(rshift_N.counts, [rshift_R.midpoints[0], rshift_N.midpoints[1]])
    else:
        raise ValueError("R and N midpoints don't match")

    rshift_phi, *_ = unfold_spectrum(
        rshift_R,
        rshift_N,
        max_iterations=2000
    )
    eng_data["rshift_N"] = rshift_N
    eng_data["rshift_phi"] = rshift_phi

### Lowpass Filtering

In [ ]:
from scipy.signal import butter, lfilter, filtfilt


def butter_lowpass(cutoff, fs, order=5):
    return butter(order, cutoff, fs=fs, btype='low', analog=False)


def butter_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    # y = lfilter(b, a, data)
    y = filtfilt(b, a, data)
    return y


order = 6
period = 0.062
fs = 1/period
cutoff = fs/4

for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts)
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts)
    lmids = lshift_phi.midpoints[1]
    wrong_side_mask = lmids <= energy_num
    wrong_side_mask = wrong_side_mask.reshape((1, -1))
    rshift_phi_counts[wrong_side_mask] = 0
    
    lshift_counts_filtered = butter_lowpass_filter(lshift_phi_counts, cutoff, fs, order)
    rshift_counts_filtered = butter_lowpass_filter(rshift_phi_counts, cutoff, fs, order)
    eng_data["lshift_phi_filtered"] = NDHistogram(lshift_counts_filtered, lshift_phi.midpoints)
    eng_data["rshift_phi_filtered"] = NDHistogram(rshift_counts_filtered, rshift_phi.midpoints)

### Peak Finding

In [ ]:
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    if energy_num in [2.356, 2.542, 2.728]:
        continue
    # print(energy)
    phi = eng_data["phi"]
    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    phi_peak_idx, *_ = find_peaks(phi.counts.reshape(-1), prominence=0.01)
    lshift_peak_idx, *_ = find_peaks(lshift_phi_filtered.counts.reshape(-1), prominence=0.01)
    rshift_peak_idx, *_ = find_peaks(rshift_phi_filtered.counts.reshape(-1), prominence=0.01)
    # print(phi_peak_idx)
    # print([phi.midpoints[1][i] for i in phi_peak_idx])
    # print([lshift_phi_filtered.midpoints[1][i] for i in lshift_peak_idx])
    # print([rshift_phi_filtered.midpoints[1][i] for i in rshift_peak_idx])
    lo_error = energy_num - lshift_phi_filtered.midpoints[1][lshift_peak_idx[0]]
    hi_error = rshift_phi_filtered.midpoints[1][rshift_peak_idx[0]] - energy_num
    e_error = (float(lo_error), float(hi_error))
    print(energy_num)
    print(e_error)
    eng_data["e_error"] = e_error

In [ ]:
all_errorbars = []
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    if energy_num in [2.356, 2.542, 2.728]:
        continue
    e_error = eng_data["e_error"]
    all_errorbars.append((energy_num, e_error))
print("Copy and paste this as horizontal (energy) error!")
print(all_errorbars)

### Plotting (Diagnostic)

In [ ]:
figsize = (15,6)
# N (with L and R shift versions)
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    # ignore values we don't want
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    print(energy)
    N = eng_data["N"]
    lshift_N = eng_data["lshift_N"]
    rshift_N = eng_data["rshift_N"]

    phi = eng_data["phi"]
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts.reshape(-1))
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts.reshape(-1))
    lmids = lshift_phi.midpoints[1]
    energy_num = float(energy[:5])
    wrong_side_mask = lmids <= energy_num
    rshift_phi_counts[wrong_side_mask] = 0

    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    fig, axs = plt.subplots(1, 3, figsize=figsize)
    ax1, ax2, ax3 = axs
    
    ax1.plot(N.midpoints[0], N.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax1.plot(lshift_N.midpoints[0], lshift_N.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax1.plot(rshift_N.midpoints[0], rshift_N.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax1.set(xlabel="L (MeVee)", ylabel="Counts", title="PHD", yscale="symlog", ylim=(1, 1e4))
    ax1.legend()

    ax2.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax2.plot(lshift_phi.midpoints[1], lshift_phi_counts, linestyle="dashed", label="Left shift")
    ax2.plot(rshift_phi.midpoints[1], rshift_phi_counts, linestyle="dashed", label="Right shift")
    ax2.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (No Filter)"
    )
    ax2.legend()

    ax3.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax3.plot(lshift_phi_filtered.midpoints[1], lshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax3.plot(rshift_phi_filtered.midpoints[1], rshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax3.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (Filtered)"
    )
    ax3.legend()

    fig.suptitle(energy)

    plt.show()

In [ ]:
figsize = (15,6)
# N (with L and R shift versions)
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    # ignore values we don't want
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    print(energy)
    N = eng_data["N"]
    lshift_N = eng_data["lshift_N"]
    rshift_N = eng_data["rshift_N"]

    phi = eng_data["phi"]
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts.reshape(-1))
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts.reshape(-1))
    lmids = lshift_phi.midpoints[1]
    energy_num = float(energy[:5])
    wrong_side_mask = lmids <= energy_num
    rshift_phi_counts[wrong_side_mask] = 0

    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    fig, axs = plt.subplots(1, 3, figsize=figsize)
    ax1, ax2, ax3 = axs
    
    ax1.plot(N.midpoints[0], N.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax1.plot(lshift_N.midpoints[0], lshift_N.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax1.plot(rshift_N.midpoints[0], rshift_N.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax1.set(xlabel="L (MeVee)", ylabel="Counts", title="PHD", yscale="symlog", ylim=(1, 1e4))
    ax1.legend()

    ax2.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax2.plot(lshift_phi.midpoints[1], lshift_phi_counts, linestyle="dashed", label="Left shift")
    ax2.plot(rshift_phi.midpoints[1], rshift_phi_counts, linestyle="dashed", label="Right shift")
    ax2.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (No Filter)"
    )
    ax2.legend()

    ax3.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax3.plot(lshift_phi_filtered.midpoints[1], lshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax3.plot(rshift_phi_filtered.midpoints[1], rshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax3.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (Filtered)"
    )
    ax3.legend()

    fig.suptitle(energy)

    plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()